# XAS from raw H5 (multi-run scan)

Build an XAS curve from a series of data runs, each taken at a different
photon energy:

1. Load a single **dark run** (x-rays off) and compute a per-bunch 2D
   background from its mean VLS spectrum.
2. Loop over the **data runs** (x-rays on). For each run, subtract the
   dark background, optionally also subtract a 1D background built from
   end-of-train no-x-ray bunches, compute per-shot VLS spectral moments,
   and accumulate per-shot vectors of `gmd`, `vls_sum`, and `vls_com`.
3. The VLS centre of mass (`vls_coms`, in source-pixel units) is taken
   as the per-shot estimate of central photon energy — once a
   pixel-to-eV calibration is available, this becomes a real eV axis.
4. Check the per-shot GMD vs integrated-VLS correlation on the combined
   dataset (a sanity check that alignment and background subtraction
   are intact).
5. Compute the scalar `XAS = sum(gmd) / sum(vls)` per run (one value
   per photon-energy point) and overall.
6. Bin every kept shot by its **VLS COM** and compute
   `sum(gmd) / sum(vls)` per bin — this is the XAS curve as a function
   of measured central photon energy.

In [ ]:
import sys
from pathlib import Path
_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import config
from data_loading import load_raw_h5
from binning import arb_bool_ar, bin_and_sum_ratio

%matplotlib inline

## Parameters

In [ ]:
# Runs --------------------------------------------------------------
DARK_RUN_NO  = 58763                       # x-rays off (dark)
DATA_RUN_NOS = [58764, 58765, 58766, 58767]  # data runs at different photon energies
MAX_FILES    = 5                           # raw H5 files per run; None = read all

# VLS pixel-space ROI around the absorption edge.
ROI = (500, 600)

# Bunches that actually saw x-rays in the data run (half-open).
# Verify against the per-bunch diagnostic plot below.
SIGNAL_BUNCH_RANGE = (0, 60)

# Optional additional 1D background built from end-of-train "off"
# bunches in the data run. Set to None to skip this step.
BG_BUNCH_RANGE = (90, 100)

# Per-shot GMD band (uJ). None = no bound on that side.
GMD_LO, GMD_HI = 0.5, None

# Number of VLS-COM bins for the XAS curve.
N_COM_BINS = 20

## 1. Dark run -> per-bunch background

Load the dark run, crop the VLS to the ROI, and average over all trains
to get a 2D `(n_bunches, n_pixels)` background. Keeping the bunch axis
lets a bunch-dependent detector baseline be removed too — which a
single 1D spectrum would smear out.

In [ ]:
dark = load_raw_h5(DARK_RUN_NO, config=2, max_files=MAX_FILES)
dark = dark.crop_vls(*ROI)
dark_bg = np.nanmean(dark.vls, axis=0)   # (n_bunches, n_pixels)
print(f"dark vls          : {dark.vls.shape}")
print(f"dark_bg           : {dark_bg.shape}  (mean over {dark.n_trains} trains)")

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.pcolormesh(dark.vls_pixels, np.arange(dark_bg.shape[0]), dark_bg,
                   cmap="inferno", shading="auto")
fig.colorbar(im, ax=ax, label="Dark intensity (arb.)")
ax.set_xlabel("Pixel")
ax.set_ylabel("Bunch index")
ax.set_title(f"Per-bunch dark background  (run {DARK_RUN_NO})")
fig.tight_layout()
plt.show()

## 2. Data runs -> dark sub + per-shot accumulation

Loop over `DATA_RUN_NOS`. For each run: load, crop the VLS to `ROI`,
subtract the 2D dark background, optionally subtract a 1D background
from `BG_BUNCH_RANGE`, compute per-shot VLS moments
(`sums`, `coms`, `widths`), slice down to `SIGNAL_BUNCH_RANGE`, and
append per-shot vectors of `gmd`, `vls_sum`, and `vls_com` to the
combined accumulators. The first run's `data` is also kept as
`example_data` for the bunch diagnostic in the next section.

In [ ]:
b_start, b_end = SIGNAL_BUNCH_RANGE
m_sig = b_end - b_start

gmd_chunks     = []
vls_sum_chunks = []
vls_com_chunks = []
run_chunks     = []
example_data   = None

for run_no in DATA_RUN_NOS:
    print(f"--- run {run_no} ---")
    d = load_raw_h5(run_no, config=2, max_files=MAX_FILES)
    d = d.crop_vls(*ROI)
    if d.n_bunches != dark_bg.shape[0]:
        raise ValueError(
            f"bunch count mismatch: run {run_no} has {d.n_bunches} "
            f"bunches, dark_bg has {dark_bg.shape[0]}."
        )
    d = d.subtract_background(dark_bg)
    if BG_BUNCH_RANGE is not None:
        d = d.auto_subtract_background(BG_BUNCH_RANGE)
    d = d.compute_vls_moments()

    gmd_chunks.append(d.gmd[:, b_start:b_end].ravel())
    vls_sum_chunks.append(d.vls_sums[:, b_start:b_end].ravel())
    vls_com_chunks.append(d.vls_coms[:, b_start:b_end].ravel())
    run_chunks.append(np.full(d.n_trains * m_sig, run_no, dtype=np.int64))

    com_mean = float(np.nanmean(d.vls_coms[:, b_start:b_end]))
    print(f"    trains={d.n_trains}, <vls_com>={com_mean:.2f} pixel")

    if example_data is None:
        example_data = d

gmd_shot     = np.concatenate(gmd_chunks)
vls_sum_shot = np.concatenate(vls_sum_chunks)
vls_com_shot = np.concatenate(vls_com_chunks)
run_shot     = np.concatenate(run_chunks)

good = (np.isfinite(gmd_shot)
        & np.isfinite(vls_sum_shot)
        & np.isfinite(vls_com_shot))
if GMD_LO is not None: good &= gmd_shot >= GMD_LO
if GMD_HI is not None: good &= gmd_shot <= GMD_HI

x_gmd = gmd_shot[good]
y_vls = vls_sum_shot[good]
c_vls = vls_com_shot[good]
r_id  = run_shot[good]

print()
print(f"shots total       : {gmd_shot.size}")
print(f"shots after filter: {x_gmd.size}")
print(f"VLS COM range     : {np.nanmin(c_vls):.2f} .. {np.nanmax(c_vls):.2f} pixel")

## 3. Mean spectrum vs bunch index (first run)

Shown for `example_data` (the first run in `DATA_RUN_NOS`). Use this to
confirm `SIGNAL_BUNCH_RANGE` (cyan dashes) actually covers the bunches
that saw x-rays, and that `BG_BUNCH_RANGE` (orange dots) lands in the
tail where there is no signal. The remaining runs are assumed to share
the same bunch layout.

In [ ]:
mean_by_bunch  = np.nanmean(example_data.vls, axis=0)      # (n_bunches, n_pixels)
total_by_bunch = np.nansum(mean_by_bunch, axis=1)          # (n_bunches,)
n_bunches = example_data.n_bunches

fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                         gridspec_kw={"width_ratios": [3, 1]}, sharey=True)
im = axes[0].pcolormesh(example_data.vls_pixels, np.arange(n_bunches),
                        mean_by_bunch, cmap="inferno", shading="auto")
fig.colorbar(im, ax=axes[0], label="Mean intensity (arb.)")
axes[0].axhline(b_start, color="cyan", lw=1, ls="--")
axes[0].axhline(b_end,   color="cyan", lw=1, ls="--")
axes[0].set_xlabel("Pixel")
axes[0].set_ylabel("Bunch index")
axes[0].set_title(f"Mean spectrum vs bunch  (run {DATA_RUN_NOS[0]})")

axes[1].plot(total_by_bunch, np.arange(n_bunches), "o-", ms=3,
             color="mediumseagreen")
axes[1].axhline(b_start, color="cyan", lw=1, ls="--", label="signal range")
axes[1].axhline(b_end,   color="cyan", lw=1, ls="--")
if BG_BUNCH_RANGE is not None:
    axes[1].axhline(BG_BUNCH_RANGE[0], color="orange", lw=1, ls=":",
                    label="bg range")
    axes[1].axhline(BG_BUNCH_RANGE[1], color="orange", lw=1, ls=":")
axes[1].set_xlabel("Integrated intensity")
axes[1].set_title("per bunch")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="lower right")
fig.tight_layout()
plt.show()

## 4. Correlation: GMD vs integrated VLS (all runs)

Per-shot, across every kept shot in every data run. A clean
near-linear correlation through the origin is the alignment / dark-sub
sanity check. If you see one cluster per run rather than a single line,
that means the integrated intensity per uJ of pulse energy varies
strongly with photon energy — i.e. there is real absorption variation,
which is what makes the XAS curve below interesting.

In [ ]:
gmd_edges = np.percentile(x_gmd, np.linspace(0, 100, 11))
gmd_cents, gmd_bool_ar = arb_bool_ar(gmd_edges, x_gmd)

vls_mean_per_gmd = np.array([
    np.nanmean(y_vls[m]) if m.any() else np.nan for m in gmd_bool_ar
])
vls_std_per_gmd  = np.array([
    np.nanstd(y_vls[m])  if m.any() else np.nan for m in gmd_bool_ar
])
gmd_mean_per_gmd = np.array([
    np.nanmean(x_gmd[m]) if m.any() else np.nan for m in gmd_bool_ar
])

r = float(np.corrcoef(x_gmd, y_vls)[0, 1])
slope, intercept = np.polyfit(x_gmd, y_vls, 1)
xl = np.linspace(gmd_mean_per_gmd.min(), gmd_mean_per_gmd.max(), 50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
h = ax1.hist2d(
    x_gmd, y_vls,
    bins=[gmd_edges,
          np.linspace(np.percentile(y_vls, 1), np.percentile(y_vls, 99), 51)],
    cmap="viridis", cmin=1, norm=mcolors.LogNorm(),
)
fig.colorbar(h[3], ax=ax1, label="shots per bin")
ax1.plot(xl, slope*xl + intercept, color="white", lw=1.5,
         label=f"slope={slope:.3g}, intercept={intercept:.3g}, r={r:.3f}")
ax1.set_xlabel("GMD (uJ)")
ax1.set_ylabel("Integrated VLS (arb.)")
ax1.set_title(f"GMD vs integrated VLS  ({x_gmd.size} shots)")
ax1.legend(loc="upper left", framealpha=0.85)

ax2.errorbar(gmd_mean_per_gmd, vls_mean_per_gmd, yerr=vls_std_per_gmd,
             fmt="o-", color="mediumseagreen", capsize=3,
             label="binned mean +/- std")
ax2.plot(xl, slope*xl + intercept, color="darkred", lw=1.2,
         label="linear fit (all shots)")
ax2.set_xlabel("GMD (uJ)")
ax2.set_ylabel("Mean integrated VLS")
ax2.set_title("Percentile-binned")
ax2.grid(alpha=0.3)
ax2.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 5. Scalar XAS per run + overall

For each data run, `XAS = sum(gmd) / sum(vls)` over every kept shot in
that run — one value per photon-energy point in the scan. The mean VLS
COM per run gives an estimate of where that run sits on the
energy axis (pixel units). The overall value across all runs is a
sanity check; it is not physically meaningful since the runs sit at
different photon energies.

In [ ]:
print(f"{'run':>8s}  {'n shots':>8s}  {'<vls_com>':>10s}  {'XAS':>10s}")
print("-" * 45)

for run_no in DATA_RUN_NOS:
    mask = r_id == run_no
    g = x_gmd[mask]
    v = y_vls[mask]
    com_mean = float(np.nanmean(c_vls[mask])) if mask.any() else float('nan')
    xas_run = (g.sum() / v.sum()) if v.sum() != 0 else float('nan')
    print(f"{run_no:8d}  {int(mask.sum()):8d}  {com_mean:10.2f}  {xas_run:10.4f}")

gmd_total = float(np.nansum(x_gmd))
vls_total = float(np.nansum(y_vls))
xas_overall = gmd_total / vls_total
print("-" * 45)
print(f"{'all':>8s}  {int(x_gmd.size):8d}  "
      f"{float(np.nanmean(c_vls)):10.2f}  {xas_overall:10.4f}")

## 6. XAS vs VLS centre of mass

Every kept shot from every run is binned by its per-shot VLS centre of
mass (`vls_coms`, in source-pixel units). Per bin,
`XAS = sum(gmd) / sum(vls)`. Percentile bins so each point on the
curve is averaged over the same number of shots.

The mean VLS-COM per run from the table above tells you which run
populates which part of the curve. Once a pixel-to-eV calibration is
available, the x axis becomes real photon energy and the absorption
edge will be at the eV expected for the sample.

In [ ]:
com_edges = np.percentile(c_vls, np.linspace(0, 100, N_COM_BINS + 1))
_, com_bool_ar = arb_bool_ar(com_edges, c_vls)

xas_per_bin, n_per_bin = bin_and_sum_ratio(com_bool_ar, x_gmd, y_vls)

com_mean_per_bin = np.array([
    np.nanmean(c_vls[m]) if m.any() else np.nan for m in com_bool_ar
])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(com_mean_per_bin, xas_per_bin, "o-",
        color="mediumseagreen", lw=1.5)
# Mark each run's mean COM so it's clear which point on the curve
# comes from which run.
for run_no in DATA_RUN_NOS:
    mask = r_id == run_no
    if mask.any():
        ax.axvline(float(np.nanmean(c_vls[mask])), color="gray",
                   ls=":", lw=0.8, alpha=0.6)
        ax.text(float(np.nanmean(c_vls[mask])),
                ax.get_ylim()[1], f" {run_no}",
                rotation=90, va="top", ha="left", fontsize=8, color="gray")
ax.set_xlabel("VLS centre of mass (pixel)")
ax.set_ylabel("XAS = sum(gmd) / sum(vls)")
ax.set_title(
    f"XAS vs VLS COM  ({len(DATA_RUN_NOS)} runs, "
    f"{N_COM_BINS} percentile bins, {x_gmd.size} shots)"
)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

print(f"{'vls_com (px)':>14s}  {'n shots':>8s}  {'XAS':>10s}")
for c, n, v in zip(com_mean_per_bin, n_per_bin, xas_per_bin):
    print(f"{c:14.2f}  {int(n):8d}  {v:10.4f}")